In [51]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import pickle
import os
import gc
from copy import deepcopy
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt
from multiprocessing import Pool

from cylinder_alignment.bundle_station import BundleStation
from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from cylinder_alignment.utils import extract_target_cylinder
from utils.write_output import write_array2txt
from utils.math_methods import calc_overlap_curve

import plotly.express as px

In [52]:
data_dir = Path('D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]

# Read extracted cylinder data

In [53]:


source_files = glob(str(data_dir / '*.e57'))
station_names = []
stations = []
for ind, fp in enumerate(source_files):
    if 'roof' in fp.lower():
        continue
    # station name
    station_name = Path(fp).stem
    # station id
    station_id = ind

    print(f'STATION {station_name}')
    station = Station(
        name=station_name, file_path=fp, read_points=False, station_id=station_id
    )
    stations.append(station)

STATION Tank409 Scan_Plains_Tank_409-_Setup_001
STATION Tank409 Scan_Plains_Tank_409-_Setup_002
STATION Tank409 Scan_Plains_Tank_409-_Setup_003
STATION Tank409 Scan_Plains_Tank_409-_Setup_004
STATION Tank409 Scan_Plains_Tank_409-_Setup_005
STATION Tank409 Scan_Plains_Tank_409-_Setup_006
STATION Tank409 Scan_Plains_Tank_409-_Setup_007
STATION Tank409 Scan_Plains_Tank_409-_Setup_008


In [54]:
source_files = glob(str(data_dir / '*.e57'))
stations = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    p = Path(fp)
    station_name = p.stem
    station_pkl_file = p.parents[0] / station_name / f'{station_name}.pkl'
    print(station_pkl_file)
    if os.path.exists(str(station_pkl_file)):
        with open(station_pkl_file, 'rb') as f:
            stations.append(pickle.load(f))


D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_001\Tank409 Scan_Plains_Tank_409-_Setup_001.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_002\Tank409 Scan_Plains_Tank_409-_Setup_002.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_003\Tank409 Scan_Plains_Tank_409-_Setup_003.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_004\Tank409 Scan_Plains_Tank_409-_Setup_004.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_005\Tank409 Scan_Plains_Tank_409-_Setup_005.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_006\Tank409 Scan_Plains_Tank_409-_Setup_006.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_007\Tank409 Scan_Plains_Tank_409-_Setup_007.pkl
D:\REGISTRATION DATA\External\JB-567 27ft EXT Le

In [55]:
# source_files = glob(str(data_dir / '*.e57'))
# station_names = []
# for fp in source_files:
#     if 'roof' in fp.lower():
#         continue
#     fn = Path(fp).stem
#     # station id
#     sn = fn[-11:-8]
#     # roof: 001 --> 008
#     # if 'roof' in fp.lower():
#     #     continue
#     # fn = Path(fp).stem
#     # # station id
#     # sn = fn[-3:]
#     print(sn)
#     station_names.append(sn)
# stations = []
# for sn in station_names:
#     pkl_fp = os.path.join(
#         data_dir,
#         f'{sn}-output',
#         f'{sn}.pkl'
#     )
#     if os.path.exists(pkl_fp):
#         with open(pkl_fp, 'rb') as f:
#             st = pickle.load(f)
#             stations.append(st)


## Removing invalid cylinders
- nan values

In [56]:
for s in stations:
    valid_cyls = []
    for c in s.refined_cylinders:
        if not np.isnan(c.radius):
            valid_cyls.append(c)
        else:
            print(s.name, c.id)
    s.refined_cylinders = valid_cyls

In [57]:
summary = []
for s in stations:
    # print('======')
    # print(s)
    s.update_filepath(data_dir, num_keep_level=1)
    for c in s.refined_cylinders:
        # update path
        if c.ptc_filepath:
            c.update_filepath(data_dir, num_keep_level=2)
        # print(c)
        summary.append({
            'station': s.name,
            'cylinder': c.id,
            'radius': c.radius
        })
summary_df = pd.DataFrame(summary)
summary_df.head()

,station,cylinder,radius
0,Tank409 Scan_Plains_Tank_409-_Setup_001,86,6.487136
1,Tank409 Scan_Plains_Tank_409-_Setup_001,89,5.279918
2,Tank409 Scan_Plains_Tank_409-_Setup_001,95,9.146709
3,Tank409 Scan_Plains_Tank_409-_Setup_002,48,6.486217
4,Tank409 Scan_Plains_Tank_409-_Setup_002,77,9.161267


In [58]:
px.scatter(
    summary_df[summary_df.radius < 50], y='radius', x='station', 
    color='station', hover_data=summary_df.columns
)

# Pairwise alignment

In [59]:
num_station = len(stations)
alignment = []
for i in range(num_station):
    st0 = stations[i]
    for j in range(i+1, num_station):
        st1 = stations[j]
        align = StationAlignment(st0, st1)
        align.align(max_delta_radius=1, top_surface=False, scan_radius=17.8, RMS_min=1)
        alignment.append(align)

Align station Tank409 Scan_Plains_Tank_409-_Setup_001 and Tank409 Scan_Plains_Tank_409-_Setup_002:
3
calling pair matching
100 is larger than the chosen RMS_min threshold = 1
3.673202955328824 is larger than the chosen RMS_min threshold = 1
100 is larger than the chosen RMS_min threshold = 1
100 is larger than the chosen RMS_min threshold = 1
3.7050769890426305 is larger than the chosen RMS_min threshold = 1
[[ 86.           6.48713589  -4.29088163 -29.98311996   0.        ]
 [ 95.           9.14670944  26.24108124  -9.96117306   0.        ]
 [ 89.           5.27991772  -0.80309939  15.99563313   0.        ]]
[[ 48.           6.48621655 -12.43410587 -30.68940544   0.        ]
 [ 77.           9.16126728  22.31437683 -41.92695236   0.        ]
 [ 92.           5.51620913  25.75377655  -4.75350475   0.        ]]
RMS = 0.043873602891648406
86 48
95 77
89 92
Align station Tank409 Scan_Plains_Tank_409-_Setup_001 and Tank409 Scan_Plains_Tank_409-_Setup_003:
3
calling pair matching
3.67813651

In [136]:
Path('')

WindowsPath('.')

In [60]:
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'wb') as f:
    pickle.dump([alignment, stations], f)

In [61]:
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
    alignment, stations = pickle.load(f)

In [62]:
ori_stations = deepcopy(stations)

# Align all stations

In [63]:
# all stations
stations = deepcopy(ori_stations)
bundle = BundleStation(stations)

## For not-connected stations which have only 2 cylinders
For each of these station: call it station i
- Determine the neighbour station j
- Analyse two possible matching between i and j
    - Apply the corresponding transformation matrix
    - Extract thin slices
    - Measure the overlap
    - The correct match is the transformation with the larger overlap

In [64]:
# bundle.handle_uncertain(
#     neighbor_info={
#         '009': '008',
#         '016': '015',
#         '018': '017'
#     }
# )

In [65]:
bundle.align_center()

# bundle.guess_target_cylinder()

[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_002'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_003'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_006'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_007'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]
[{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_006'}, {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}]


In [66]:
bundle.cluster_cylinder()

In [67]:
for s in bundle.stations:
    s.ptc = None

In [68]:
with open(os.path.join(data_dir, 'tank_info.pkl'), 'wb') as f:
    pickle.dump(bundle.tanks_info, f)

In [69]:
with open(os.path.join(data_dir, 'bundle.pkl'), 'wb') as f:
    pickle.dump(bundle, f)

In [70]:
with open(os.path.join(data_dir, 'bundle.pkl'), 'rb') as f:
    bundle = pickle.load(f)
    stations = bundle.stations

# Visual inspection

To identify group id that represents the target cylinder 

In [105]:
z_cut = 3
thickness = 0.01
cross_sections = []
for tank_label, cyls in bundle.tanks_info['tanks'].items():
    tank_id = int(tank_label)
    print('tank', tank_id)
    for c in cyls:
        print(c.ptc_filepath)
        c.read_points()
        print(c.points.shape)
        c.transform(c.station.align['rotation'], c.station.align['translation'])
        z_ground = c.find_ground_z()
        z = z_cut + z_ground
        ind = (z <= c.points[:,2]) & (c.points[:,2] <= z + thickness)
        c.points = c.points[ind,:]
        p = np.c_[c.points, tank_id * np.ones([c.points.shape[0],1])]
        cross_sections.append(p)
        del c.points
        c.points = None
        gc.collect()

tank 0
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_001\Tank409 Scan_Plains_Tank_409-_Setup_001 cluster_86.pcd
(1756759, 3)
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_002\Tank409 Scan_Plains_Tank_409-_Setup_002 cluster_48.pcd
(1134930, 3)
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_003\Tank409 Scan_Plains_Tank_409-_Setup_003 cluster_77.pcd
(819635, 3)
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_004\Tank409 Scan_Plains_Tank_409-_Setup_004 cluster_142.pcd
(711287, 3)
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_005\Tank409 Scan_Plains_Tank_409-_Setup_005 cluster_74.pcd
(1480647, 3)
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_006\Tank409 Scan_Plains_Tank_409-_Setup_006 cluster_77.pcd
(1445038, 3)
D:\REGISTRATION DATA\External\JB-5

In [106]:
cs_points = np.concatenate(cross_sections, axis=0)
cs_df = pd.DataFrame(cs_points, columns=['x', 'y', 'z', 'tank_id'])
cs_df['tank_label'] = cs_df.tank_id.apply(lambda i: str(int(i)))
cs_df.head()

,x,y,z,tank_id,tank_label
0,-10.601177,-31.588636,-6.898644,0.0,0
1,-10.571458,-31.693790,-6.891561,0.0,0
2,-10.637149,-31.439919,-6.895632,0.0,0
3,-10.625990,-31.495165,-6.898026,0.0,0
4,-10.623197,-31.502691,-6.889484,0.0,0


In [88]:
def visualise_thin_slice(df):
    x_min = df.x.min()
    x_max = df.x.max()
    delta_x = (x_max - x_min) * 0.2
    y_min = df.y.min()
    y_max = df.y.max()
    delta_y = (y_max - y_min) * 0.2
    fig = px.scatter(df, x='x', y='y', color='tank_label', range_x=[x_min-delta_x, x_max+delta_x], range_y=[y_min-delta_y, y_max+delta_y])
    # fig = px.scatter_3d(df, x='x', y='y', z='z', color='station_name', range_x=[x_min-delta_x, x_max+delta_x], range_y=[y_min-delta_y, y_max+delta_y])
    fig.update_layout(
        width = 800,
        height = 800,
        title = "thin slice"
    )
    fig.update_yaxes(
        scaleanchor = "x",
        scaleratio = 1,
    )
    return fig

In [125]:
center_df = bundle.tanks_info['centers_xy']
center_df.sort_values('label')
center_df.rename(columns={'label': 'tank_label'}, inplace=True)

In [133]:
df = pd.concat([cs_df, center_df])
df = df.drop(['station', 'cylinder id', 'radius', 'xy'], axis=1)
# df.fillna(0, inplace=True)

In [134]:
fig = visualise_thin_slice(df)
fig.show()

In [ ]:
center_df = bundle.tanks_info['centers_xy']
center_df.sort_values('label')

In [ ]:
center_df.groupby('label').size()

In [ ]:
center_df[center_df.label == '3']

In [22]:
px.scatter(
    center_df, 
    x='x', y='y', 
    color='label', hover_data=['station', 'cylinder id', 'radius', 'x', 'y'], 
    width=700, height=700)

# Cluster id of the target tank

In [39]:
tank_id = 3

# Cylinder vertical axes alignment

In [61]:
target_tank = bundle.tanks_info['tanks'][str(tank_id)]

In [62]:
for cyl in target_tank:
    print(cyl.station.name, cyl.ptc_filepath)

Tank409 Scan_Plains_Tank_409-_Setup_004 D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_004\Tank409 Scan_Plains_Tank_409-_Setup_004 cluster_138.pcd
Tank409 Scan_Plains_Tank_409-_Setup_005 D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_005\Tank409 Scan_Plains_Tank_409-_Setup_005 cluster_63.pcd
Tank409 Scan_Plains_Tank_409-_Setup_006 D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_006\Tank409 Scan_Plains_Tank_409-_Setup_006 cluster_76.pcd
Tank409 Scan_Plains_Tank_409-_Setup_007 D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_007\Tank409 Scan_Plains_Tank_409-_Setup_007 cluster_41.pcd
Tank409 Scan_Plains_Tank_409-_Setup_008 D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\Tank409 Scan_Plains_Tank_409-_Setup_008\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_73.pcd


In [58]:
ext_write_dir = os.path.join(data_dir, f'tank_{tank_id}_ext_ground_original')

In [26]:
if not os.path.exists(ext_write_dir):
    os.mkdir(ext_write_dir)

num_process = 8
with Pool(num_process) as p:
    p.starmap(
        extract_target_cylinder,
        zip(target_tank, [ext_write_dir] * len(target_tank))
    )

{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_001'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_002'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_003'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_004'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_005'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_006'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_007'}
{'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}


Run the following cell to load original ground-extended data

In [63]:
for c in target_tank:
    c.update_filepath(new_dir=ext_write_dir, num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
    print(c.ptc_filepath)
                # c.read_points()

D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3_ext_ground_original\Tank409 Scan_Plains_Tank_409-_Setup_004 cluster_138.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3_ext_ground_original\Tank409 Scan_Plains_Tank_409-_Setup_005 cluster_63.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3_ext_ground_original\Tank409 Scan_Plains_Tank_409-_Setup_006 cluster_76.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3_ext_ground_original\Tank409 Scan_Plains_Tank_409-_Setup_007 cluster_41.pcd
D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3_ext_ground_original\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_73.pcd


# Calculate the alignment matrices for all target cylinders

In [66]:
bundle.calc_align_matrix(f'{tank_id}')

calculating alignment matrix for tank 3
Tank409 Scan_Plains_Tank_409-_Setup_004 cluster 138
Tank409 Scan_Plains_Tank_409-_Setup_005 cluster 63
Tank409 Scan_Plains_Tank_409-_Setup_006 cluster 76
Tank409 Scan_Plains_Tank_409-_Setup_007 cluster 41
Tank409 Scan_Plains_Tank_409-_Setup_008 cluster 73


In [68]:
out_dir = os.path.join(data_dir, f'tank_{tank_id}', 'ext_ground')
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
bundle.write_tank(
    tank_label=f'{tank_id}',
    write_dir=out_dir,
    read_points=True,
    data_type='pcd'
)

writing cylinder 3
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3\ext_ground\Tank409 Scan_Plains_Tank_409-_Setup_004.pcd: (11630129, 3) points
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3\ext_ground\Tank409 Scan_Plains_Tank_409-_Setup_005.pcd: (7052308, 3) points
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3\ext_ground\Tank409 Scan_Plains_Tank_409-_Setup_006.pcd: (5890824, 3) points
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3\ext_ground\Tank409 Scan_Plains_Tank_409-_Setup_007.pcd: (5189934, 3) points
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\tank_3\ext_ground\Tank409 Scan_Plains_Tank_409-_Setup_008.pcd: (39636729, 3) points


# Thin slice

In [70]:
s.refined_cylinders

[{'station': {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}, 'radius': 4.209363, 'axis': "{'position': array([[ 1.38061953e-03],\n       [-4.30146581e-04],\n       [ 8.89745746e+00]]), 'vector': array([[8.67361738e-19],\n       [0.00000000e+00],\n       [1.00000001e+00]])}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-567 27ft EXT Leica\\tank_3_ext_ground_original\\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_73.pcd'},
 {'station': {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}, 'radius': 6.488218, 'axis': "{'position': array([ 25.629889  , -36.01658   ,  -0.14610104], dtype=float32), 'vector': array([ 0.00327348, -0.00168231,  0.9999932 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-567 27ft EXT Leica\\Tank409 Scan_Plains_Tank_409-_Setup_008\\Tank409 Scan_Plains_Tank_409-_Setup_008 cluster_82.pcd'},
 {'station': {'name': 'Tank409 Scan_Plains_Tank_409-_Setup_008'}, 'radius': 9.199898, 'axis': "{'position': array([ 60.72507   , -25.780779

In [46]:
# a = np.random.randint(0,10,[4,3])
a = np.array([
    [0.2, 0.4, 0],
    [0.5, 0.2, 0],
    [0.5, 1.3, 0],
    [1.3, 1.4, 0],
    [1.5, 1.2, 0]
])
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(a)

In [27]:
pcd.voxel_down_sample_and_trace

<bound method PyCapsule.voxel_down_sample_and_trace of PointCloud with 4 points.>

In [48]:
d = pcd.voxel_down_sample(voxel_size=1)

In [49]:
np.asarray(pcd.points)

array([[0.2, 0.4, 0. ],
       [0.5, 0.2, 0. ],
       [0.5, 1.3, 0. ],
       [1.3, 1.4, 0. ],
       [1.5, 1.2, 0. ]])

In [50]:
np.asarray(d.points)

array([[0.35, 0.3 , 0.  ],
       [0.5 , 1.3 , 0.  ],
       [1.4 , 1.3 , 0.  ]])

In [73]:
write_dir = os.path.join(data_dir, 'thin_slice')
if not os.path.exists(write_dir):
    os.mkdir(write_dir)


thickness = 0.01
step = 20

z_lower = 3
z_higher = 6
cnt = []

for cyl in target_tank:
    print(cyl.station.name, 'cluster', cyl.id)
    s = cyl.station
    s.read_cylinders()
    # transform
    s.transform(cyl.align['rotation'], cyl.align['translation'])
    print(f'z_min = {s.ptc[:,2].min()}')
    # once slice
    slice_cnt = []
    i = np.ones(s.ptc.shape[0]) == 0
    for z in np.arange(z_lower, z_higher, step):
        i = i | ((z <= s.ptc[:,2]) & (s.ptc[:,2] <= z + thickness))
        slice_cnt.append(i.sum())
    s.ptc = s.ptc[i,:]
    s.write_points(os.path.join(write_dir, f'{s.name}.pcd'), data_type='pcd')
    del s.ptc
    gc.collect()

Tank409 Scan_Plains_Tank_409-_Setup_004 cluster 138
z_min = 0.0
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_004.pcd
Tank409 Scan_Plains_Tank_409-_Setup_005 cluster 63
z_min = 0.0
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_005.pcd
Tank409 Scan_Plains_Tank_409-_Setup_006 cluster 76
z_min = 0.0
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_006.pcd
Tank409 Scan_Plains_Tank_409-_Setup_007 cluster 41
z_min = 0.0
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_007.pcd
Tank409 Scan_Plains_Tank_409-_Setup_008 cluster 73
z_min = 1.7763568394002505e-15
write to D:\REGISTRATION DATA\External\JB-567 27ft EXT Leica\thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_008.pcd


In [ ]:
# bundle.write_all_tanks(data_dir, surrounding=False)

# Align stations after 2D alignment

In [16]:
file_path = os.path.join(data_dir, 'thin_slice', 'transform_2d.pkl')
with open(file_path, 'rb') as f:
    transform_matrix = pickle.load(f)

In [ ]:
tank_id = 2
# transform the entire stations or just the target cylinder
entire_station = False

if entire_station:
    write_dir = os.path.join(data_dir, 'align_2d_station')
    if not os.path.exists(write_dir):
        os.mkdir(write_dir)
else:
    write_dir=os.path.join(data_dir, 'tank_8', 'align_2d')
    if not os.path.exists(write_dir):
        os.makedirs(write_dir)

for s, transf in zip(stations, transform_matrix):
    print(s)
    if entire_station:
        s.read_cylinders()
        # rough alignment phase 1
        s.transform(s.align['rotation'], s.align['translation'])
        # rough alignment phase 2
        s.transform(transf['rotation'], transf['translation'])
        # write
        s.write_points(os.path.join(write_dir, f'{s.name}.txt'))
    else:
        for c in s.refined_cylinders:
            try:
                # print(c.id)
                if c.tank_id == tank_id:
                    # # rough alignment phase 1
                    # c.transform(s.align['rotation'], s.align['translation'])
                    c.update_filepath(os.path.join(data_dir, 'tank_8', 'ext_ground'), num_keep_level=1, file_name=f'{s.name}.txt')
                    c.read_points()
                    # rough alignment phase 2
                    c.transform(transf['rotation'], transf['translation'])
                    # write
                    c.write_points(write_dir=write_dir, filename=f'{s.name}.txt')
                    del c.points
                    gc.collect()
            except Exception as e:
                print(e)
                pass


In [40]:
import open3d as o3d

In [ ]:
o3d.pipelines.registration.evaluate_registration